# Milestone 1

## Imports

In [898]:
import nltk
import re
import pandas as pd
import glob
from nltk.corpus import stopwords
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
# nltk.download('punkt')
# nltk.download('punkt_tab')
# nltk.download('stopwords')

## Load Datasets

### QA Dataset

In [899]:
import os

path = "data/QA"
files = os.listdir(path)

for f in files:
    with open(os.path.join(path, f), encoding="utf-8") as file:
        text = file.read()
        print(f, len(text))

0urc3PabvOs_QA.csv 39181
3AwL93uolIA_QA.csv 34114
8rcY34IA-6I_QA.csv 34706
8ZL2AAxQmLQ_QA.csv 42284
ArytJ_HZ-1E_QA.csv 42329
DbuoaakWh7g_QA.csv 35242
gv5hDK2YQfA_QA.csv 33161
IxsJEffQAZA_QA.csv 35730
MOEwXtL2DQ4_QA.csv 37415
nc8oJTETqoI_QA.csv 37153
NJ-JcDcff8o_QA.csv 38688
yc7x5jNhXIQ_QA.csv 34614
z2NGnjXG5uQ_QA.csv 40717


### Transcripts Dataset

In [900]:
import os

path = "data/Transcripts"
files = os.listdir(path)

for f in files:
    with open(os.path.join(path, f), encoding="utf-8") as file:
        text = file.read()
        print(f, len(text))

أعظم طائرة حربية  الدحيح.txt 38222
الأخطبوط  الدحيح.txt 38054
الساموراي  الدحيح.txt 34077
تاج محل  الدحيح.txt 26431
جون كينيدي  الدحيح.txt 56962
فيزياء و فلسفة الحركة  الدحيح.txt 47095
كيف تحولت روسيا إلى إمبراطورية؟  الدحيح.txt 59094
كيف تسيطر على عقول البشر؟  الدحيح.txt 45570
كيف تنقل جبل وزنه 30 طن قبل أن يغرق؟  الدحيح.txt 38574
مصير الأرض و الشمس و كل شيء  الدحيح.txt 33909
معركة ذي قار  الدحيح.txt 51260
منابع النيل  الدحيح.txt 49836
هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح.txt 35210


### All QA loadded into single DataFrame

In [901]:
files = glob.glob("data/QA/*.csv")

dfs = []
for f in files:
    temp = pd.read_csv(f)
    # temp["source_file"] = f
    dfs.append(temp)

df_train = pd.concat(dfs, ignore_index=True)

## Text Exploration

### Trancripts Inspection

In [902]:
from collections import Counter

path = "data/Transcripts"
files = os.listdir(path)

for f in files:
    with open(os.path.join(path, f), encoding="utf-8") as file:
        text = file.read()
        words = text.split()
        freq = Counter(words)
        
        print(freq.most_common(20))

[('يا', 142), ('في', 128), ('عزيزي،', 120), ('اللي', 102), ('على', 90), ('من', 88), ('ما', 83), ('إن', 67), ('دا', 45), ('مش', 41), ('دي', 36), ('هو', 36), ('انت', 33), ('طيارة', 32), ('الطيارة', 30), ('عشان', 29), ('كل', 29), ('كان', 28), ('احنا', 23), ('الـF-35', 21)]
[('يا', 134), ('في', 122), ('عزيزي،', 94), ('ما', 85), ('إن', 82), ('من', 82), ('على', 68), ('الأخطبوط', 65), ('اللي', 64), ('هو', 44), ('دا', 42), ('زي', 39), ('مش', 38), ('كدا،', 34), ('انت', 31), ('كل', 31), ('عشان', 31), ('بس', 28), ('لمّا', 26), ('فيه', 25)]
[('يا', 124), ('في', 113), ('عزيزي،', 110), ('كان', 65), ('من', 63), ('اللي', 63), ('الساموراي', 57), ('إن', 56), ('ما', 49), ('على', 33), ('كانوا', 31), ('دا', 30), ('أو', 29), ('كانت', 29), ('عشان', 28), ('زي', 25), ('مش', 24), ('هُما', 21), ('كدا،', 21), ('هو', 19)]
[('في', 96), ('يا', 91), ('عزيزي،', 67), ('من', 66), ('اللي', 64), ('ما', 50), ('على', 47), ('إن', 39), ('دا', 32), ('مش', 31), ('كان', 28), ('كل', 24), ('زي', 22), ('دي', 22), ('"تاج', 21), ('"ش

### QA Inspection

In [903]:
df_train.head()

,video_id,video_title,question_id,question,answer,difficulty
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,ماذا ورد في هذا الموضع من النص؟,"افتح موضوع جديد يا ""ميدو""، أنا مش ناقص!",Easy
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,ما الجملة المذكورة في هذا السياق؟,وبعد كدا، هتنطفي هي كمان.,Medium
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,كيف صيغت العبارة هنا؟,- أيوة.,Easy
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,ما الذي قيل في هذا الجزء؟,- عشان كُل حاجة بتنتهي.,Medium
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,ما النص الحرفي المذكور في هذه الفقرة؟,- عشان الفيزيا بتقول كدا.,Easy


In [904]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 3890 entries, 0 to 3889
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   video_id     3890 non-null   str  
 1   video_title  3890 non-null   str  
 2   question_id  3890 non-null   str  
 3   question     3890 non-null   str  
 4   answer       3890 non-null   str  
 5   difficulty   3890 non-null   str  
dtypes: str(6)
memory usage: 182.5 KB


In [905]:
df_train.describe()

,video_id,video_title,question_id,question,answer,difficulty
count,3890,3890,3890,3890,3890,3890
unique,13,13,3890,310,3468,2
top,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,ماذا ورد في هذا الموضع من النص؟,احنا دلوقتي، يا عزيزي، في يوليو 2011.,Easy
freq,300,300,1,600,30,2005


In [906]:
df_train.isna().sum()

video_id       0
video_title    0
question_id    0
question       0
answer         0
difficulty     0
dtype: int64

In [907]:
df_train.head()

,video_id,video_title,question_id,question,answer,difficulty
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,ماذا ورد في هذا الموضع من النص؟,"افتح موضوع جديد يا ""ميدو""، أنا مش ناقص!",Easy
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,ما الجملة المذكورة في هذا السياق؟,وبعد كدا، هتنطفي هي كمان.,Medium
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,كيف صيغت العبارة هنا؟,- أيوة.,Easy
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,ما الذي قيل في هذا الجزء؟,- عشان كُل حاجة بتنتهي.,Medium
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,ما النص الحرفي المذكور في هذه الفقرة؟,- عشان الفيزيا بتقول كدا.,Easy


In [908]:
all_words = " ".join(df_train["answer"].astype(str)).split()

freq = Counter(all_words)

for word, count in freq.most_common(50):
    print(f"{word}: {count}")

في: 718
اللي: 410
من: 393
ما: 261
على: 244
إن: 208
يا: 152
كان: 134
هو: 98
مش: 94
زي: 91
كل: 89
عشان: 88
فيها: 82
غير: 75
دا: 70
كانت: 67
عزيزي،: 67
هي: 66
احنا: 63
جدًا: 62
عن: 60
انت: 60
ولاية: 60
"تكساس".: 60
الطيارة: 60
دي: 57
مع: 56
اسمها: 49
هذا: 48
وهو: 48
أو: 47
أنا: 46
بشكل: 45
كدا،: 44
هذه: 42
العالم: 42
حاجة: 41
-: 40
ممكن: 40
إنه: 40
أي: 38
إلى: 37
دلوقتي،: 37
الأمريكي: 37
بس: 36
فيه: 33
ولا: 33
واقف: 32
أقدم: 32


## Data Preprocessing

In [909]:
df_train["question_length"] = df_train["question"].apply(len)
df_train["answer_length"] = df_train["answer"].apply(len)

print(df_train.describe())

       question_length  answer_length
count      3890.000000    3890.000000
mean         29.481234      27.937532
std           5.950017       8.209206
min          14.000000       3.000000
25%          25.000000      21.000000
50%          31.000000      28.000000
75%          33.000000      34.000000
max          38.000000      52.000000


### OLD and GOLD function

In [910]:
# from pathlib import Path
# from difflib import get_close_matches

# TRANSCRIPTS_DIR = Path("data/Transcripts")

# def normalize_arabic_text(text: str) -> str:
#     text = str(text).strip().lower()
#     # remove zero width characters
#     text = re.sub(r'[\u200B-\u200D\uFEFF]', '', text)
#     # remove Arabic diacritics
#     text = re.sub(r'[\u0617-\u061A\u064B-\u0652]', '', text)
#     # normalize common Arabic letter variants
#     text = re.sub(r"[إأآا]", "ا", text)
#     text = re.sub(r"ى", "ي", text)
#     text = re.sub(r"ة", "ه", text)
#     text = re.sub(r"گ", "ك", text)
#     text = text.replace("ـ", "")
#     return text

# def remove_timestamps(text):
#     return re.sub(r'\d+\.\d+:', '', text)

# def normalize_title(title: str) -> str:
#     t = normalize_arabic_text(title)
#     t = t.replace(".txt", "")
#     t = re.sub(r"\bالدحيح\b", "", t)
#     t = re.sub(r"\s+", " ", t).strip()
#     return t

# def tokenize_words(text: str):
#     # keep Arabic/English word tokens only
#     return re.findall(r"[\w]+", str(text), flags=re.UNICODE)

# def normalize_tokens(tokens):
#     normalized = []
#     for tok in tokens:
#         nt = normalize_arabic_text(tok)
#         nt = re.sub(r"[^\w]", "", nt, flags=re.UNICODE)
#         if nt:
#             normalized.append(nt)
#     return normalized

# def find_sublist_index(haystack, needle):
#     if not needle or len(needle) > len(haystack):
#         return -1
#     n = len(needle)
#     for i in range(len(haystack) - n + 1):
#         if haystack[i:i+n] == needle:
#             return i
#     return -1

# # Build transcript index keyed by normalized title
# transcript_index = {}
# for transcript_path in TRANSCRIPTS_DIR.glob("*.txt"):
#     raw_text = transcript_path.read_text(encoding="utf-8")
#     raw_text = remove_timestamps(raw_text)
#     original_tokens = tokenize_words(raw_text)
#     normalized_tokens = normalize_tokens(original_tokens)

#     key = normalize_title(transcript_path.stem)
#     transcript_index[key] = {
#         "path": transcript_path,
#         "original_tokens": original_tokens,
#         "normalized_tokens": normalized_tokens
#     }

# def match_transcript_key(video_title: str):
#     key = normalize_title(video_title)
#     if key in transcript_index:
#         return key
#     # fallback fuzzy match if title is slightly different
#     matches = get_close_matches(key, list(transcript_index.keys()), n=1, cutoff=0.55)
#     return matches[0] if matches else None

# def extract_context_from_row(row, window=20):
#     matched_key = match_transcript_key(row["video_title"])
#     if not matched_key:
#         return ""

#     answer_tokens = tokenize_words(str(row["answer"]))
#     answer_tokens_norm = normalize_tokens(answer_tokens)
#     if not answer_tokens_norm:
#         return ""

#     transcript = transcript_index[matched_key]
#     idx = find_sublist_index(transcript["normalized_tokens"], answer_tokens_norm)
#     if idx == -1:
#         return ""

#     start = max(0, idx - window)
#     end = min(len(transcript["original_tokens"]), idx + len(answer_tokens_norm) + window)
#     # keep context in original (non-normalized) transcript form
#     return " ".join(transcript["original_tokens"][start:end])

# # Add context column
# df_train["context"] = df_train.apply(extract_context_from_row, axis=1)

# # Optional quick quality check
# found_ratio = (df_train["context"].str.len() > 0).mean() * 100
# print(f"Context extracted for {found_ratio:.1f}% of rows")
# df_train[["video_title", "answer", "context"]].head()

### Clean And normalize Text Fields

In [911]:
def process_text_column(df, column_name):

    df[column_name] =df[column_name].apply(remove_stopwords_and_punctuation_from_text)
    df[column_name] =df[column_name].apply(normalize_text)
    return df

def remove_stopwords_and_punctuation_from_text(text):
    nltk_stop_words = set(stopwords.words('arabic'))


    # punctuation ='!"$%&()*,-./:;<=>?@[\\]^_`{|}~'
    punctuation = [
    "،","؛","؟","ـ","«","»","‹","›","“","”","‘","’",
    ".",",",";",":","!","?","-","_","(",")","[","]","{","}",
    "\"","'","/","\\","|","@","#","$","%","^","&","*","+","=","<",">","~","`","``","''"
    ]
    for p in punctuation:
        text = text.replace(p, '')

    tokens = nltk.word_tokenize(text)

    filtered_tokens = [word for word in tokens if word not in nltk_stop_words]
    # Join the tokens back into a string
    filtered_text = ' '.join(filtered_tokens)
    return filtered_text

def remove_timestamps(text):
    return re.sub(r'\d+\.\d+:', '', text)

def normalize_text(text):
    
    # 1. Convert English letters to lowercase
    text = text.lower()
    
    # 2. Remove Arabic Tashkeel (diacritics)
    tashkeel = r'[\u0617-\u061A\u064B-\u0652]'
    text = re.sub(tashkeel, '', text)

    # 3. Normalize Arabic letters
    text = re.sub("[إأآا]", "ا", text)
    text = re.sub("ى", "ي", text)
    # text = re.sub("ؤ", "و", text)
    # text = re.sub("ئ", "ي", text)
    text = re.sub("ة", "ه", text)
    text = re.sub("گ", "ك", text)

    # 4. Remove Tatweel (ـ)
    text = re.sub("ـ", "", text)

    return text

In [912]:
process_text_column(df_train, "question")
process_text_column(df_train, "answer")

,video_id,video_title,question_id,question,answer,difficulty,question_length,answer_length
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,الموضع النص,افتح موضوع جديد ميدو مش ناقص,Easy,31,39
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,الجمله المذكوره السياق,وبعد كدا هتنطفي كمان,Medium,33,25
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,صيغت العباره,ايوه,Easy,21,7
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,قيل الجزء,عشان كل حاجه بتنتهي,Medium,25,23
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,النص الحرفي المذكور الفقره,عشان الفيزيا بتقول كدا,Easy,37,25
...,...,...,...,...,...,...,...,...
3885,z2NGnjXG5uQ,كيف تحولت روسيا إلى إمبراطورية؟ | الدحيح,z2NGnjXG5uQ_Q296,الموضع النص,تشارلز افتكر بطرس هرب المعركه,Medium,31,40
3886,z2NGnjXG5uQ,كيف تحولت روسيا إلى إمبراطورية؟ | الدحيح,z2NGnjXG5uQ_Q297,الجمله المذكوره السياق,ويخش ياخد روسيا,Easy,33,18
3887,z2NGnjXG5uQ,كيف تحولت روسيا إلى إمبراطورية؟ | الدحيح,z2NGnjXG5uQ_Q298,صيغت العباره,اخطر عنصر الاتحاد,Medium,21,23
3888,z2NGnjXG5uQ,كيف تحولت روسيا إلى إمبراطورية؟ | الدحيح,z2NGnjXG5uQ_Q299,قيل الجزء,بولندا الخصم الصعب العنيف,Easy,25,29


### Add Context Column

In [913]:
from difflib import get_close_matches


def add_context_column(df_train, transcripts_dir="data/Transcripts", window=20):

    # Load transcripts
    transcripts = {}

    files = glob.glob(os.path.join(transcripts_dir, "*.txt"))

    for file_path in files:

        file_name = os.path.basename(file_path).replace(".txt", "")

        with open(file_path, encoding="utf-8") as f:
            text = f.read()

        text = remove_timestamps(text)
        text = remove_stopwords_and_punctuation_from_text(text)
        text = normalize_text(text)

        transcripts[file_name] = text

    transcript_names = list(transcripts.keys())
    contexts = []


    # Iterate rows
    for _, row in df_train.iterrows():

        title = row["video_title"]
        answer = normalize_text(row["answer"])

        # find closest transcript name
        match = get_close_matches(title, transcript_names, n=1, cutoff=0.4)

        if not match:
            contexts.append("")
            continue

        transcript = transcripts[match[0]]

        idx = transcript.find(answer)

        if idx == -1:
            contexts.append("")
            continue

        before = transcript[:idx].split()
        after = transcript[idx + len(answer):].split()

        start = max(0, len(before) - window)

        context_words = before[start:] + answer.split() + after[:window]

        context = " ".join(context_words)

        contexts.append(context)


    # add column
    df_train["context"] = contexts

    return df_train


df_train = add_context_column(df_train)
df_train.head()

,video_id,video_title,question_id,question,answer,difficulty,question_length,answer_length,context
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,الموضع النص,افتح موضوع جديد ميدو مش ناقص,Easy,31,39,بابا بابا بابا احنا جينا ازاي لا دا السؤال بتا...
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,الجمله المذكوره السياق,وبعد كدا هتنطفي كمان,Medium,33,25,لا دا السؤال بتاع امبارح افتح موضوع جديد ميدو ...
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,صيغت العباره,ايوه,Easy,21,7,موضوع جديد ميدو مش ناقص طيب الشمس هتفضل منوره ...
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,قيل الجزء,عشان كل حاجه بتنتهي,Medium,25,23,ميدو مش ناقص طيب الشمس هتفضل منوره لحد امتي لح...
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,النص الحرفي المذكور الفقره,عشان الفيزيا بتقول كدا,Easy,37,25,هتفضل منوره لحد امتي لحد نهايه الكون وبعد كدا ...


### Check answer and question length after preprocessing

In [914]:
df_train["answer_length"] = df_train["answer"].apply(len)
df_train.head()

,video_id,video_title,question_id,question,answer,difficulty,question_length,answer_length,context
0,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q001,الموضع النص,افتح موضوع جديد ميدو مش ناقص,Easy,31,28,بابا بابا بابا احنا جينا ازاي لا دا السؤال بتا...
1,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q002,الجمله المذكوره السياق,وبعد كدا هتنطفي كمان,Medium,33,20,لا دا السؤال بتاع امبارح افتح موضوع جديد ميدو ...
2,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q003,صيغت العباره,ايوه,Easy,21,4,موضوع جديد ميدو مش ناقص طيب الشمس هتفضل منوره ...
3,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q004,قيل الجزء,عشان كل حاجه بتنتهي,Medium,25,19,ميدو مش ناقص طيب الشمس هتفضل منوره لحد امتي لح...
4,0urc3PabvOs,مصير الأرض و الشمس و كل شيء | الدحيح,0urc3PabvOs_Q005,النص الحرفي المذكور الفقره,عشان الفيزيا بتقول كدا,Easy,37,22,هتفضل منوره لحد امتي لحد نهايه الكون وبعد كدا ...


### Check on short answers

In [915]:
df_train[df_train["answer_length"] == 5]

,video_id,video_title,question_id,question,answer,difficulty,question_length,answer_length,context
496,3AwL93uolIA,الساموراي | الدحيح,3AwL93uolIA_Q197,الجمله المذكوره الموضع,اتنين,Easy,33,5,العالم عندهم حاجات زي الساموراي كدا ناس متمرنه...
640,8rcY34IA-6I,الأخطبوط | الدحيح,8rcY34IA-6I_Q041,النص الفقره رقم 41,نخدره,Easy,37,5,نقول دا دا عبقري احنا عزيزي كنا بنتعامل الاخطب...
649,8rcY34IA-6I,الأخطبوط | الدحيح,8rcY34IA-6I_Q050,النص الفقره رقم 50,عزيزي,Medium,37,5,لا خلاص هاقوم اسلقه حالا اخلص الfile اعزائي ال...
711,8rcY34IA-6I,الأخطبوط | الدحيح,8rcY34IA-6I_Q112,النص الفقره رقم 112,يقولك,Medium,38,5,زاهيه اوي انت غالبا كنت سام فالاخطبوط ممكن يلع...
897,8ZL2AAxQmLQ,كيف تنقل جبل وزنه 30 طن قبل أن يغرق؟ | الدحيح,8ZL2AAxQmLQ_Q008,صيغت العباره,حرامي,Medium,21,5,استاذ سيد استاذنك حاجه تبقاش تمشي بدري انت يوم...
1228,ArytJ_HZ-1E,هل Citizen Kane أفضل فيلم في التاريخ؟ | الدحيح,ArytJ_HZ-1E_Q039,قيل الجزء,زاندو,Easy,25,5,خدت الذكاء وخدت مني الجمال ليه كدا لساني سهله ...
1229,ArytJ_HZ-1E,هل Citizen Kane أفضل فيلم في التاريخ؟ | الدحيح,ArytJ_HZ-1E_Q040,النص الحرفي المذكور الفقره,زبادو,Medium,37,5,الذكاء وخدت مني الجمال ليه كدا لساني سهله اهي ...
1337,ArytJ_HZ-1E,هل Citizen Kane أفضل فيلم في التاريخ؟ | الدحيح,ArytJ_HZ-1E_Q148,صيغت العباره,الوقت,Medium,21,5,وبطاله وحاجه نيله السينما مكان بنلجاله عشان نه...
2679,MOEwXtL2DQ4,فيزياء و فلسفة الحركة | الدحيح,MOEwXtL2DQ4_Q290,النص الحرفي المذكور الفقره,الوقت,Medium,37,5,دول جماعه سايبين اللي وراهم واللي قدامهم ومشغو...
2711,nc8oJTETqoI,منابع النيل | الدحيح,nc8oJTETqoI_Q022,الجمله المذكوره السياق,طبقيص,Medium,33,5,يجيب دول عندنا الفريزر كتير بحيث كده بقي انا ع...


### Remove short answers and Drop unrelated columns 

In [916]:
# remove rows with very short answers (likely to be noise) like df_train[df_train["answer_length"] < 6 ]

df_train = df_train[df_train["answer_length"] >= 6].reset_index(drop=True)

# drop unrelated columns to save memory
df_train = df_train.drop(columns=["video_id", "video_title", "question_id"])


### Analyze text after preprocessing

In [917]:
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 3849 entries, 0 to 3848
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   question         3849 non-null   str  
 1   answer           3849 non-null   str  
 2   difficulty       3849 non-null   str  
 3   question_length  3849 non-null   int64
 4   answer_length    3849 non-null   int64
 5   context          3849 non-null   str  
dtypes: int64(2), str(4)
memory usage: 180.6 KB


In [918]:
df_train.head()

,question,answer,difficulty,question_length,answer_length,context
0,الموضع النص,افتح موضوع جديد ميدو مش ناقص,Easy,31,28,بابا بابا بابا احنا جينا ازاي لا دا السؤال بتا...
1,الجمله المذكوره السياق,وبعد كدا هتنطفي كمان,Medium,33,20,لا دا السؤال بتاع امبارح افتح موضوع جديد ميدو ...
2,قيل الجزء,عشان كل حاجه بتنتهي,Medium,25,19,ميدو مش ناقص طيب الشمس هتفضل منوره لحد امتي لح...
3,النص الحرفي المذكور الفقره,عشان الفيزيا بتقول كدا,Easy,37,22,هتفضل منوره لحد امتي لحد نهايه الكون وبعد كدا ...
4,الموضع النص,عشان الفيزيا رياضه تطبيقيه,Medium,31,26,نهايه الكون وبعد كدا هتنطفي كمان الكون هينتهي ...


In [919]:
all_words = " ".join(df_train["answer"].astype(str)).split()

freq = Counter(all_words)

for word, count in freq.most_common(50):
    print(f"{word}: {count}")

اللي: 410
دا: 114
عزيزي: 98
مش: 96
دي: 93
جدا: 92
زي: 92
عشان: 89
كدا: 75
احنا: 67
كانت: 67
انت: 63
حاجه: 61
تكساس: 61
ولايه: 60
الطياره: 60
العالم: 57
اسمها: 49
بشكل: 45
دلوقتي: 45
سنه: 44
الامريكي: 42
ممكن: 41
الله: 38
عادي: 34
مارتن: 34
ضوئيه: 33
النيل: 33
لانه: 33
الناس: 32
واقف: 32
اقدم: 32
صعب: 32
7: 32
لحد: 31
المعبد: 31
هتغير: 31
2011: 30
لحضرتك: 30
فخر: 30
الطيران: 30
الf35: 30
لوكهيد: 30
هيجرب: 30
الرادار: 30
يلقطها: 30
لغات: 30
المقدم: 30
ايريك: 30
سميث: 30


In [920]:
# Save the processed DataFrame to a new CSV file
# df_train.to_csv("output.csv", index=False, encoding="utf-8-sig")

In [921]:
# Save processed text
# output_path = "processed_transcript.txt"
# with open(output_path, "w", encoding="utf-8") as file:
#     file.write(processed_text)

## Tokenizer

In [922]:
tokenizer = Tokenizer(
    num_words=20000,
    split=' ',
    char_level=False,
    oov_token="<OOV>"
  )

In [ ]:
# fit on data input to tokenizer
all_texts = df_train['question'] + ' ' + df_train['context'] + ' ' + df_train['answer']  # optionally include answers in tokenizer fit
tokenizer.fit_on_texts(all_texts)
input_sequences = tokenizer.texts_to_sequences(df_train['question'] + ' ' + df_train['context'])

# fit on output
answer_sequences = tokenizer.texts_to_sequences(df_train['answer'])

In [924]:
# use this to get insight on the Tokenizer
vocab_list = tokenizer.word_index

In [925]:
from sklearn.model_selection import train_test_split

df["input_text"] = df["question"] + " " + df["context"]

X_train, X_test, y_train, y_test = train_test_split(
    df["input_text"],
    df["answer"],
    test_size=0.2,
    random_state=42
)

train_sequences = tokenizer.texts_to_sequences(X_train)

KeyError: 'context'

### Padding

In [ ]:
# # Padding
trunc_type='post'
padding_type='post'
max_length=62000

train_padded = pad_sequences(
    train_sequences,
    maxlen=max_length,
    padding=padding_type,
    truncating=trunc_type
)

### Tokenizer Visualizations

In [ ]:
import matplotlib.pyplot as plt

# --- 2. Vocabulary stats ---
total_vocab = len(vocab_list)
capped_vocab = min(total_vocab, 20000)
print(f"Total unique tokens (full vocab):  {total_vocab}")
print(f"Vocab size used (num_words cap):   {capped_vocab}")

# Top 30 most frequent words (lowest index = highest frequency)
top_n = 30
top_words = sorted(vocab_list.items(), key=lambda x: x[1])[:top_n]
words_labels = [w for w, _ in top_words]
word_indices = [i for _, i in top_words]

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(words_labels, word_indices, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Token Index (lower index = more frequent)')
ax.set_title(f'Top {top_n} Most Frequent Tokens in Vocabulary')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

# --- 3. Sequence length distribution (before padding) ---
seq_lengths = [len(s) for s in train_sequences]

print(f"Sequence lengths — min: {min(seq_lengths)}, max: {max(seq_lengths)}, "
      f"mean: {np.mean(seq_lengths):.1f}, median: {np.median(seq_lengths):.1f}")
print(f"% sequences <= max_length ({max_length}): "
      f"{100 * np.mean(np.array(seq_lengths) <= max_length):.1f}%")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_lengths, bins=40, color='teal', edgecolor='white')
ax.axvline(max_length, color='red', linestyle='--', label=f'max_length = {max_length}')
ax.axvline(np.mean(seq_lengths), color='orange', linestyle='--',
           label=f'mean = {np.mean(seq_lengths):.1f}')
ax.set_xlabel('Sequence Length (tokens)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Sequence Lengths Before Padding')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# --- 4. Padding zero-ratio + sample decode ---
zero_ratio = (train_padded == 0).sum() / train_padded.size
print(f"Padded matrix shape:  {train_padded.shape}")
print(f"Zero (pad) tokens:    {zero_ratio*100:.1f}% of all positions")

# Pie chart: padding vs real tokens  |  Line chart: token IDs of first sample
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].pie(
    [1 - zero_ratio, zero_ratio],
    labels=['Real tokens', 'Padding zeros'],
    autopct='%1.1f%%',
    colors=['steelblue', 'lightgray'],
    startangle=90
)
axes[0].set_title('Padding vs Real Tokens')

sample_seq = train_padded[0]
non_pad = sample_seq[sample_seq != 0]
axes[1].plot(range(len(non_pad)), non_pad, marker='o', markersize=3,
             linewidth=0.8, color='steelblue')
axes[1].set_xlabel('Token Position')
axes[1].set_ylabel('Token ID')
axes[1].set_title('Token IDs for Sample[0] (non-pad tokens)')

plt.tight_layout()
plt.show()

# Decode sample back to words
reverse_vocab = {v: k for k, v in vocab_list.items()}
decoded = [reverse_vocab.get(int(t), '<OOV>') for t in non_pad[:20]]
print("\nFirst 20 decoded tokens of sample[0]:")
print(' '.join(decoded))